# 3.23 — Bagging & Random Forests

Bagging trains many deliberately noisy trees on perturbed versions of the data, then averages their predictions so the noise cancels while the reusable signal remains. Random forests add one more source of diversity — random feature choices at each split — so the averaged trees are less correlated and the ensemble becomes a more stable decision rule.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build bagging and random forests one idea at a time. Run each cell in order and read the printed intermediate values — every bootstrap sample, split, average, variance calculation, and validation score is made visible. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, bootstrap indices, random numbers, and small numerical checks.
import matplotlib.pyplot as plt  # visualizations for trees, averages, and validation curves.
np.random.seed(0)  # reproducibility for every bootstrap and synthetic data set.

### 1. Bootstrap samples: make many training sets from one training set

Bagging begins by replacing one fixed training table with many **bootstrap** tables. A bootstrap table has the same size as the original table, but each row is sampled **with replacement**. Some rows appear twice or three times; other rows are left out. That deliberate perturbation makes different trees see different empirical risks, so their errors are not identical.

In [ ]:
x_w = np.arange(8)  # eight original row ids; each id stands for one training example.
rng_w = np.random.default_rng(0)  # local generator so this walkthrough is reproducible.
boot_w = rng_w.integers(0, len(x_w), size=len(x_w))  # sample n row ids with replacement.
counts_w = np.bincount(boot_w, minlength=len(x_w))  # count how many times each original row appears.

print("bootstrap indices:", boot_w)  # inspect the perturbed training set.
print("row counts:", counts_w)  # inspect duplicates and omissions.

assert boot_w.shape == x_w.shape  # bootstrap sample has the original sample size.

▶ What you'll see: a length-8 list with repeated row ids and at least one omitted row.

In [ ]:
oob_w = np.where(counts_w == 0)[0]  # rows not selected by this bootstrap are out-of-bag.

print("out-of-bag row ids:", oob_w)  # inspect rows available for validation of this tree.
print("unique rows used:", int(np.sum(counts_w > 0)), "of", len(x_w))  # how much of the original set was touched.

assert len(oob_w) > 0  # for this seed, some rows are left out.

▶ What you'll see: bootstrap perturbation creates a built-in validation slice for the tree that did not see those rows.

In [ ]:
plt.figure(figsize=(5, 3))  # create a compact count plot.
plt.bar(x_w, counts_w, color="steelblue")  # draw how often each original row entered the bootstrap set.
plt.axhline(1, color="black", linestyle="--", linewidth=1)  # one copy is the original-table reference.
plt.title("1: bootstrap row multiplicities")  # title the visualization.
plt.xlabel("original row id")  # label the original examples.
plt.ylabel("copies in bootstrap")  # label the sampled multiplicity.
plt.show()  # display the plot.

▶ What you'll see: bars above 1 are duplicated rows; bars at 0 are out-of-bag rows.

*Why it's done this way:* sampling with replacement simulates new training sets drawn from the same empirical distribution. A single high-variance tree changes a lot when its training rows change; bagging exploits that sensitivity by producing many different trees whose average is less volatile than any one tree.

### 2. Bagging average: the ensemble prediction is a mean

Once the bootstrap trees are trained, bagging predicts by a plain average: $$\hat f_{bag}(x)=\frac1B\sum_{b=1}^B\hat f_b(x).$$ The numerator adds the votes or numeric predictions; the denominator turns the sum back into the same prediction scale. Averaging is not a cosmetic choice — independent noise cancels at rate roughly $1/B$.

In [ ]:
preds_w = np.array([2.9, 3.4, 2.7, 3.2, 3.0])  # five tree predictions for the same x.
bag_pred_w = float(np.mean(preds_w))  # bagging prediction = arithmetic average.

print("tree predictions:", preds_w)  # inspect noisy individual trees.
print("bagged prediction:", round(bag_pred_w, 3))  # inspect the ensemble mean.

assert round(bag_pred_w, 3) == 3.04  # verify the hand-checkable average.

▶ What you'll see: individual trees bounce around, while the ensemble prediction sits near their center.

In [ ]:
centered_w = preds_w - bag_pred_w  # deviations around the ensemble mean.

print("deviations from ensemble mean:", np.round(centered_w, 3))  # inspect cancellation.
print("sum of deviations:", round(float(np.sum(centered_w)), 10))  # must be zero for a mean.

assert abs(np.sum(centered_w)) < 1e-10  # arithmetic check: the mean balances deviations.

▶ What you'll see: positive and negative deviations cancel exactly around the average.

In [ ]:
plt.figure(figsize=(5, 3))  # create a compact prediction plot.
plt.scatter(np.arange(len(preds_w)), preds_w, color="gray", label="trees")  # show individual trees.
plt.axhline(bag_pred_w, color="crimson", linewidth=2, label="bagged mean")  # show the ensemble.
plt.title("2: bagging averages noisy predictions")  # title the plot.
plt.xlabel("tree id")  # label tree index.
plt.ylabel("prediction")  # label predicted value.
plt.legend()  # explain markers and line.
plt.show()  # display the plot.

▶ What you'll see: the red line is less extreme than most single-tree predictions.

*Why it's done this way:* if each tree prediction is signal plus zero-mean noise, the signal appears in every tree but the noise points in different directions. The average preserves the common signal and shrinks the unstable part, which is exactly the variance-reduction promise of bagging.

### 3. Variance reduction depends on correlation, not just the number of trees

Averaging $B$ independent predictions with variance $\sigma^2$ gives variance $\sigma^2/B$. Trees are not fully independent because they share the same original data, so a useful rule of thumb is $$\operatorname{Var}(\bar T)=\sigma^2\left(\rho+\frac{1-\rho}{B}\right),$$ where $\rho$ is average correlation between tree errors. More trees help, but highly correlated trees keep a variance floor.

In [ ]:
sigma2_w = 1.0  # use unit single-tree variance so the formula is easy to read.
B_grid_w = np.array([1, 2, 5, 10, 50, 100])  # compare small and large forests.
rho_w = 0.2  # assume tree errors are moderately correlated.
var_w = sigma2_w * (rho_w + (1 - rho_w) / B_grid_w)  # correlated-average variance formula.

print("B values:", B_grid_w)  # inspect ensemble sizes.
print("variance factors:", np.round(var_w, 3))  # inspect remaining variance vs one tree.

assert round(float(var_w[3]), 3) == 0.28  # with B=10 and rho=.2, variance factor is .28.

▶ What you'll see: variance falls quickly at first, then approaches the correlation floor 0.2.

In [ ]:
rho_grid_w = np.array([0.0, 0.2, 0.6])  # independent, moderately correlated, highly correlated trees.
var_by_rho_w = np.array([r + (1 - r) / 50 for r in rho_grid_w])  # fixed B=50 comparison.

print("rho values:", rho_grid_w)  # inspect correlations.
print("variance at B=50:", np.round(var_by_rho_w, 3))  # inspect the cost of correlation.

assert np.all(var_by_rho_w >= rho_grid_w)  # finite B cannot beat the correlation floor.

▶ What you'll see: a 50-tree ensemble is excellent when rho is 0, but much less magical when rho is 0.6.

In [ ]:
plt.figure(figsize=(5, 3))  # create a variance curve figure.
for r_w in rho_grid_w:  # draw one curve per correlation level.
    plt.plot(B_grid_w, r_w + (1 - r_w) / B_grid_w, marker="o", label=f"rho={r_w}")  # variance factor.
plt.title("3: tree correlation limits bagging")  # title the plot.
plt.xlabel("number of trees B")  # label ensemble size.
plt.ylabel("variance factor")  # label relative variance.
plt.legend()  # show correlation labels.
plt.show()  # display the curves.

▶ What you'll see: all curves improve with more trees, but high-correlation curves flatten high.

*Why it's done this way:* bagging attacks variance by averaging, but duplicated training rows still make trees resemble each other. Random forests therefore add feature randomness to reduce $\rho$, because lowering correlation can matter as much as increasing $B$.

### 4. A tiny decision stump: split by minimizing squared error

A tree is built from splits. In regression, a simple split chooses a threshold and predicts the mean target on each side. The split score is the mean squared error after replacing every target by its side's mean. This is empirical risk in its smallest tree-shaped form.

In [ ]:
X_w = np.array([0.05, 0.12, 0.20, 0.55, 0.70, 0.90])[:, None]  # one feature, six rows.
y_w = np.array([0.2, 0.1, 0.3, 1.2, 1.1, 1.4])  # target jumps upward after about .5.
thresholds_w = (X_w[:-1, 0] + X_w[1:, 0]) / 2  # candidate thresholds between sorted x values.

print("candidate thresholds:", np.round(thresholds_w, 3))  # inspect split candidates.

assert len(thresholds_w) == 5  # six ordered rows give five between-row thresholds.

▶ What you'll see: each threshold is a possible place to make a two-leaf stump.

In [ ]:
scores_w = []  # store one MSE per threshold.
for t_w in thresholds_w:  # evaluate every split point.
    left_w = y_w[X_w[:, 0] <= t_w]  # targets on the left leaf.
    right_w = y_w[X_w[:, 0] > t_w]  # targets on the right leaf.
    pred_w = np.where(X_w[:, 0] <= t_w, np.mean(left_w), np.mean(right_w))  # side means.
    scores_w.append(np.mean((y_w - pred_w) ** 2))  # empirical risk for this stump.
best_idx_w = int(np.argmin(scores_w))  # choose the lowest-risk threshold.
best_t_w = float(thresholds_w[best_idx_w])  # read the winning threshold.

print("split MSEs:", np.round(scores_w, 3))  # inspect all risks.
print("best threshold:", round(best_t_w, 3), "best MSE:", round(float(scores_w[best_idx_w]), 3))  # inspect winner.

assert round(best_t_w, 3) == 0.375  # the split between low and high target groups wins.

▶ What you'll see: the threshold between 0.20 and 0.55 has the smallest squared error.

In [ ]:
plt.figure(figsize=(5, 3))  # create a stump visualization.
plt.scatter(X_w[:, 0], y_w, color="black", label="data")  # plot training points.
plt.axvline(best_t_w, color="crimson", linestyle="--", label="best split")  # show chosen threshold.
plt.title("4: stump split chosen by MSE")  # title the plot.
plt.xlabel("feature x")  # label feature.
plt.ylabel("target y")  # label target.
plt.legend()  # show split label.
plt.show()  # display the plot.

▶ What you'll see: the split separates the low-response group from the high-response group.

*Why it's done this way:* a leaf mean is the squared-error minimizer for values inside that leaf. Trying thresholds and choosing the lowest post-split MSE is therefore the tree's local ERM step: it asks which split makes each leaf as internally homogeneous as possible.

### 5. Bag many stumps and validate with out-of-bag rows

Every bootstrap tree leaves out some rows. Those out-of-bag rows can score that tree without a separate validation split. Averaging OOB predictions over trees that did not train on a row gives a cheap estimate of future behavior.

In [ ]:
def stump_fit_w(X, y, rows):  # fit a one-feature regression stump on selected rows.
    xs, ys = X[rows, 0], y[rows]  # bootstrap subset.
    order = np.argsort(xs)  # sort so thresholds are between neighbors.
    xs, ys = xs[order], ys[order]  # ordered feature and targets.
    ts = (xs[:-1] + xs[1:]) / 2  # candidate thresholds.
    best = (np.inf, ts[0], np.mean(ys), np.mean(ys))  # score, threshold, left mean, right mean.
    for t in ts:  # scan thresholds.
        m = xs <= t  # left-side mask.
        if np.sum(m) == 0 or np.sum(~m) == 0:  # require two nonempty leaves.
            continue  # skip degenerate split.
        pred = np.where(m, np.mean(ys[m]), np.mean(ys[~m]))  # leaf-mean predictions.
        mse = np.mean((ys - pred) ** 2)  # training MSE for this threshold.
        if mse < best[0]:  # keep the best split.
            best = (mse, t, np.mean(ys[m]), np.mean(ys[~m]))  # update best stump.
    return best[1:]  # threshold and two leaf means.

def stump_predict_w(model, X):  # predict from a fitted stump.
    t, left, right = model  # unpack threshold and leaf values.
    return np.where(X[:, 0] <= t, left, right)  # choose a leaf by threshold.

▶ What you'll see: no output yet — we defined the minimal tree machinery used by the ensemble.

In [ ]:
B_w = 25  # number of bootstrap stumps.
oob_sum_w = np.zeros(len(y_w))  # accumulate OOB predictions per row.
oob_count_w = np.zeros(len(y_w))  # count how many trees were OOB for each row.
models_w = []  # store fitted stumps for later averaging.
for b_w in range(B_w):  # train each bootstrap stump.
    rows_w = rng_w.integers(0, len(y_w), size=len(y_w))  # bootstrap rows.
    model_w = stump_fit_w(X_w, y_w, rows_w)  # fit stump on bootstrap sample.
    models_w.append(model_w)  # save the model.
    left_out_w = np.setdiff1d(np.arange(len(y_w)), np.unique(rows_w))  # OOB rows for this tree.
    if len(left_out_w):  # only score rows left out by this bootstrap.
        oob_sum_w[left_out_w] += stump_predict_w(model_w, X_w[left_out_w])  # add OOB predictions.
        oob_count_w[left_out_w] += 1  # record OOB coverage.

print("OOB counts per row:", oob_count_w.astype(int))  # inspect validation coverage.

assert np.all(oob_count_w > 0)  # every row received at least one OOB prediction for this seed.

▶ What you'll see: each row is validated by a subset of trees that did not train on it.

In [ ]:
oob_pred_w = oob_sum_w / oob_count_w  # average OOB predictions for each row.
oob_mse_w = float(np.mean((y_w - oob_pred_w) ** 2))  # OOB validation MSE.
train_pred_w = np.mean([stump_predict_w(m_w, X_w) for m_w in models_w], axis=0)  # full bagged training prediction.
train_mse_w = float(np.mean((y_w - train_pred_w) ** 2))  # apparent training MSE.

print("train MSE:", round(train_mse_w, 3), "OOB MSE:", round(oob_mse_w, 3))  # compare fit and validation.

assert oob_mse_w >= 0 and train_mse_w >= 0  # both are valid losses.

▶ What you'll see: OOB MSE is the more honest number because each row is predicted by trees that skipped it.

In [ ]:
plt.figure(figsize=(5, 3))  # create a prediction comparison.
plt.scatter(X_w[:, 0], y_w, color="black", label="truth")  # original observations.
plt.scatter(X_w[:, 0], train_pred_w, color="seagreen", label="bagged train pred")  # ensemble predictions.
plt.scatter(X_w[:, 0], oob_pred_w, color="orange", marker="x", label="OOB pred")  # OOB predictions.
plt.title("5: bagged stumps with OOB validation")  # title the plot.
plt.xlabel("x")  # label feature.
plt.ylabel("y")  # label target.
plt.legend()  # show prediction types.
plt.show()  # display the plot.

▶ What you'll see: OOB predictions are noisier than in-bag ensemble predictions but estimate unseen performance better.

*Why it's done this way:* the raw training number can be flattering because each tree helped fit many rows it later predicts. OOB scoring restores the train-versus-future distinction from generalization theory without needing extra data.

### 6. Random forests: randomize features to lower tree correlation

Random forests modify bagging by letting each split inspect only a random subset of features. That can force trees to use different explanations of the data, lowering the correlation term from concept 3. The average remains the same formula, but the base learners become more diverse.

In [ ]:
X2_w = np.array([[0.0, 0.2], [0.1, 0.4], [0.2, 0.1], [0.8, 0.7], [0.9, 0.8], [1.0, 0.6]])  # two-feature toy data.
y2_w = np.array([0., 0., 0., 1., 1., 1.])  # class labels with a clean high-vs-low pattern.
features_w = np.arange(X2_w.shape[1])  # available feature ids.
chosen_w = rng_w.choice(features_w, size=1, replace=False)  # random subset at a split.

print("features available:", features_w)  # inspect all features.
print("feature offered to this split:", chosen_w)  # inspect random feature subset.

assert len(chosen_w) == 1  # mtry=1 for this tiny random-forest demo.

▶ What you'll see: a split sees only one feature, even though the data has two.

In [ ]:
def best_stump_feature_w(X, y, feature_ids):  # choose best stump among the offered features.
    best = (np.inf, feature_ids[0], 0.0, 0.0, 0.0)  # score, feature, threshold, left mean, right mean.
    for j in feature_ids:  # scan only allowed features.
        order = np.argsort(X[:, j])  # sort by this feature.
        xs, ys = X[order, j], y[order]  # ordered values.
        ts = (xs[:-1] + xs[1:]) / 2  # thresholds.
        for t in ts:  # evaluate thresholds.
            m = xs <= t  # left mask.
            if np.sum(m) == 0 or np.sum(~m) == 0:  # avoid empty leaves.
                continue  # skip invalid split.
            pred = np.where(m, np.mean(ys[m]), np.mean(ys[~m]))  # leaf probabilities.
            mse = np.mean((ys - pred) ** 2)  # impurity as squared error for probabilities.
            if mse < best[0]:  # keep best allowed split.
                best = (mse, j, t, np.mean(ys[m]), np.mean(ys[~m]))  # update.
    return best  # return split details.

split_all_w = best_stump_feature_w(X2_w, y2_w, np.array([0, 1]))  # ordinary bagged tree split.
split_rf_w = best_stump_feature_w(X2_w, y2_w, chosen_w)  # random-forest split with feature subset.

print("best split using all features:", split_all_w[:3])  # inspect unrestricted choice.
print("best split using offered feature:", split_rf_w[:3])  # inspect randomized choice.

▶ What you'll see: the random-forest split is the best split under a feature constraint, not necessarily the globally best split.

In [ ]:
losses_w = np.array([0.235, 0.135, 0.488])  # verified per-example losses from the lesson prose.
R_S_w = float(np.mean(losses_w))  # empirical average loss.
cost_w = 0.100  # method cost or regularization guardrail.
score_w = R_S_w + cost_w  # decision score.
flex_w = 0.434  # tempting more-flexible alternative from the lesson prose.
stable_w = 0.80 * score_w  # stabilizing knob reduces decision score by 20%.

print("R_S:", round(R_S_w, 3), "score:", round(score_w, 3), "stable:", round(stable_w, 3))  # inspect decision numbers.
print("relative gap vs flexible:", round((flex_w - score_w) / flex_w, 3))  # compare on a meaningful scale.

assert round(R_S_w, 3) == 0.286 and round(score_w, 3) == 0.386 and round(stable_w, 3) == 0.309  # verified lesson arithmetic.

▶ What you'll see: the stabilized score is lowest, matching the lesson's end-to-end decision arithmetic.

In [ ]:
plt.figure(figsize=(5, 3))  # create final decision plot.
plt.bar(["baseline", "flexible", "stabilized"], [score_w, flex_w, stable_w], color=["gray", "orange", "seagreen"])  # compare decision scores.
plt.title("6: compare full decision scores")  # title the plot.
plt.ylabel("lower score is better")  # label selection criterion.
plt.show()  # display the plot.

▶ What you'll see: the stabilized method has the shortest bar, so it is the one carried forward in this toy case.

*Why it's done this way:* random feature subsets are a constraint that trades a little single-tree greediness for a lot of ensemble diversity. The right comparison is the full decision score — raw fit plus cost, gap, and stability — because a flexible tree that wins training loss can still lose once variance and validation are counted.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses small
> numbers, prints the intermediate values, draws one picture, and ends with an `assert`.

### ✍️ Toy 1 · Bootstrap sampling creates duplicates and OOB rows

A bootstrap sample has the original size, but replacement makes some rows repeat and others disappear.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_rows = np.arange(8)

print("original row ids:", t1_rows.tolist())  # -> [0, 1, 2, 3, 4, 5, 6, 7]

t1_bootstrap = t1_rng.integers(0, len(t1_rows), size=len(t1_rows))

print("bootstrap row ids:", t1_bootstrap.tolist())  # -> [6, 5, 4, 2, 2, 0, 0, 0]

t1_counts = np.bincount(t1_bootstrap, minlength=len(t1_rows))

print("row multiplicities:", t1_counts.tolist())  # -> [3, 0, 2, 0, 1, 1, 1, 0]

t1_oob = np.where(t1_counts == 0)[0]

print("out-of-bag rows:", t1_oob.tolist())  # -> [1, 3, 7]

assert t1_bootstrap.size == t1_rows.size

plt.figure(figsize=(5, 2.8))
plt.bar(t1_rows, t1_counts, color="steelblue")
plt.axhline(1, color="black", linestyle="--", linewidth=1)
plt.xlabel("row id")
plt.ylabel("copies")
plt.title("Toy 1 · bootstrap multiplicities")
plt.show()

▶ What you'll see: rows `1`, `3`, and `7` have zero copies, so they are out-of-bag for this tree.

### ✍️ Toy 2 · Bagging predictions are an arithmetic mean

The ensemble prediction keeps the shared signal by averaging noisy tree predictions for the same point.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_preds = np.array([2.0, 2.4, 1.8, 2.2, 2.1, 2.5])

print("tree predictions:", t2_preds.tolist())  # -> [2.0, 2.4, 1.8, 2.2, 2.1, 2.5]

t2_sum = np.sum(t2_preds)

print("prediction sum:", round(float(t2_sum), 3))  # -> 13.0

t2_count = len(t2_preds)

print("number of trees:", t2_count)  # -> 6

t2_mean = t2_sum / t2_count

print("bagged mean:", round(float(t2_mean), 3))  # -> 2.167

t2_deviations = t2_preds - t2_mean

print("deviations:", np.round(t2_deviations, 3).tolist())  # -> [-0.167, 0.233, -0.367, 0.033, -0.067, 0.333]

assert round(float(t2_mean), 3) == 2.167

plt.figure(figsize=(4.8, 2.8))
plt.scatter(np.arange(t2_count), t2_preds, color="gray", label="trees")
plt.axhline(t2_mean, color="crimson", linewidth=2, label="mean")
plt.xlabel("tree id")
plt.ylabel("prediction")
plt.title("Toy 2 · ensemble mean")
plt.legend()
plt.show()

▶ What you'll see: the red mean sits between the individual noisy tree predictions.

### ✍️ Toy 3 · Correlation sets a variance floor

More trees reduce variance quickly at first, but correlated tree errors leave a floor near `rho`.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_B = np.array([1, 2, 4, 8, 16, 32])

print("tree counts B:", t3_B.tolist())  # -> [1, 2, 4, 8, 16, 32]

t3_rho = 0.25

print("correlation rho:", t3_rho)  # -> 0.25

t3_single_var = 1.0

print("single-tree variance:", t3_single_var)  # -> 1.0

t3_factor = t3_single_var * (t3_rho + (1 - t3_rho) / t3_B)

print("ensemble variance factors:", np.round(t3_factor, 3).tolist())  # -> [1.0, 0.625, 0.438, 0.344, 0.297, 0.273]

assert t3_factor[-1] > t3_rho

plt.figure(figsize=(4.8, 2.8))
plt.plot(t3_B, t3_factor, marker="o", color="purple")
plt.axhline(t3_rho, color="black", linestyle="--", label="rho floor")
plt.xlabel("number of trees")
plt.ylabel("variance factor")
plt.title("Toy 3 · correlation limits averaging")
plt.legend()
plt.show()

▶ What you'll see: the curve falls toward `0.25` but cannot go below the correlation floor.

### ✍️ Toy 4 · A regression stump picks the lowest-MSE threshold

A one-split regression tree predicts the mean on each side and chooses the split with lowest squared error.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_x = np.array([0, 1, 2, 3, 4, 5, 6, 7])

print("feature values:", t4_x.tolist())  # -> [0, 1, 2, 3, 4, 5, 6, 7]

t4_y = np.array([1, 1, 2, 2, 5, 5, 6, 6], dtype=float)

print("targets:", t4_y.tolist())  # -> [1.0, 1.0, 2.0, 2.0, 5.0, 5.0, 6.0, 6.0]

t4_thresholds = (t4_x[:-1] + t4_x[1:]) / 2

print("candidate thresholds:", t4_thresholds.tolist())  # -> [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5]

t4_mse = []
for t4_threshold in t4_thresholds:
    t4_left = t4_x <= t4_threshold
    t4_left_mean = np.mean(t4_y[t4_left])
    t4_right_mean = np.mean(t4_y[~t4_left])
    t4_pred = np.where(t4_left, t4_left_mean, t4_right_mean)
    t4_mse.append(np.mean((t4_y - t4_pred) ** 2))
t4_mse = np.array(t4_mse)

print("split MSEs:", np.round(t4_mse, 3).tolist())  # -> [3.214, 2.417, 1.5, 0.25, 1.5, 2.417, 3.214]

t4_best_index = int(np.argmin(t4_mse))

print("best index:", t4_best_index)  # -> 3

t4_best_threshold = t4_thresholds[t4_best_index]

print("best threshold:", float(t4_best_threshold))  # -> 3.5

t4_best_mse = t4_mse[t4_best_index]

print("best MSE:", round(float(t4_best_mse), 3))  # -> 0.25

assert round(float(t4_best_mse), 3) == 0.25

plt.figure(figsize=(4.8, 2.8))
plt.plot(t4_thresholds, t4_mse, marker="o", color="teal")
plt.axvline(t4_best_threshold, color="crimson", linestyle="--")
plt.xlabel("threshold")
plt.ylabel("MSE")
plt.title("Toy 4 · stump threshold search")
plt.show()

▶ What you'll see: the split between `3` and `4` separates the low and high target groups best.

### ✍️ Toy 5 · Out-of-bag predictions average only skipped trees

For each row, OOB validation uses predictions from trees whose bootstrap sample did not include that row.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_y = np.array([1.0, 1.2, 1.1, 4.0, 4.2, 3.9])

print("true targets:", t5_y.tolist())  # -> [1.0, 1.2, 1.1, 4.0, 4.2, 3.9]

t5_bootstrap = np.array([[0, 1, 1, 3, 4, 4], [0, 2, 2, 3, 3, 5], [1, 2, 4, 4, 5, 5], [0, 0, 1, 3, 5, 5]])

print("bootstrap rows:\n", t5_bootstrap)  # -> [[0 1 1 3 4 4] [0 2 2 3 3 5] [1 2 4 4 5 5] [0 0 1 3 5 5]]

t5_tree_preds = np.array([[1.1, 1.1, 1.1, 4.1, 4.1, 4.1], [1.15, 1.15, 1.15, 3.95, 3.95, 3.95], [1.2, 1.2, 1.2, 4.05, 4.05, 4.05], [1.05, 1.05, 1.05, 4.0, 4.0, 4.0]])

print("tree predictions:\n", t5_tree_preds)  # -> [[1.1  1.1  1.1  4.1  4.1  4.1 ] [1.15 1.15 1.15 3.95 3.95 3.95] [1.2  1.2  1.2  4.05 4.05 4.05] [1.05 1.05 1.05 4.   4.   4.  ]]

t5_inbag = np.array([[t5_i in t5_row for t5_i in range(len(t5_y))] for t5_row in t5_bootstrap])

print("in-bag mask:\n", t5_inbag.astype(int))  # -> [[1 1 0 1 1 0] [1 0 1 1 0 1] [0 1 1 0 1 1] [1 1 0 1 0 1]]

t5_oob = ~t5_inbag

print("OOB mask:\n", t5_oob.astype(int))  # -> [[0 0 1 0 0 1] [0 1 0 0 1 0] [1 0 0 1 0 0] [0 0 1 0 1 0]]

t5_oob_count = np.sum(t5_oob, axis=0)

print("OOB counts:", t5_oob_count.tolist())  # -> [1, 1, 2, 1, 2, 1]

t5_oob_sum = np.sum(t5_tree_preds * t5_oob, axis=0)

print("OOB prediction sums:", np.round(t5_oob_sum, 3).tolist())  # -> [1.2, 1.15, 2.15, 4.05, 7.95, 4.1]

t5_oob_pred = t5_oob_sum / t5_oob_count

print("OOB predictions:", np.round(t5_oob_pred, 3).tolist())  # -> [1.2, 1.15, 1.075, 4.05, 3.975, 4.1]

t5_train_pred = np.mean(t5_tree_preds, axis=0)

print("all-tree predictions:", np.round(t5_train_pred, 3).tolist())  # -> [1.125, 1.125, 1.125, 4.025, 4.025, 4.025]

t5_oob_mse = np.mean((t5_y - t5_oob_pred) ** 2)

print("OOB MSE:", round(float(t5_oob_mse), 3))  # -> 0.023

t5_train_mse = np.mean((t5_y - t5_train_pred) ** 2)

print("all-tree MSE:", round(float(t5_train_mse), 3))  # -> 0.011

assert np.all(t5_oob_count > 0)

plt.figure(figsize=(4.8, 2.8))
plt.scatter(np.arange(len(t5_y)), t5_y, color="black", label="truth")
plt.scatter(np.arange(len(t5_y)), t5_oob_pred, marker="x", color="orange", label="OOB")
plt.scatter(np.arange(len(t5_y)), t5_train_pred, color="seagreen", label="all trees")
plt.xlabel("row")
plt.ylabel("prediction")
plt.title("Toy 5 · OOB averaging")
plt.legend()
plt.show()

▶ What you'll see: OOB predictions are computed row by row from only the trees that skipped each row.

### ✍️ Toy 6 · Random forests restrict features at a split

A random feature subset can force the split to use the best available feature instead of the global best.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_features = np.array([0, 1, 2])

print("all feature ids:", t6_features.tolist())  # -> [0, 1, 2]

t6_gains = np.array([0.50, 0.30, 0.20])

print("global feature gains:", t6_gains.tolist())  # -> [0.5, 0.3, 0.2]

t6_selected = t6_rng.choice(t6_features, size=2, replace=False)

print("features offered:", t6_selected.tolist())  # -> [1, 2]

t6_available_gains = t6_gains[t6_selected]

print("offered gains:", t6_available_gains.tolist())  # -> [0.3, 0.2]

t6_best_local_index = int(np.argmax(t6_available_gains))

print("best offered index:", t6_best_local_index)  # -> 0

t6_best_feature = int(t6_selected[t6_best_local_index])

print("best offered feature:", t6_best_feature)  # -> 1

t6_global_best = int(np.argmax(t6_gains))

print("global best feature:", t6_global_best)  # -> 0

assert t6_best_feature != t6_global_best

plt.figure(figsize=(4.8, 2.8))
plt.bar(["x0", "x1", "x2"], t6_gains, color=["lightgray", "seagreen", "seagreen"])
plt.ylabel("gain")
plt.title("Toy 6 · feature subset changes the menu")
plt.show()

▶ What you'll see: feature `0` has the largest global gain but is unavailable, so feature `1` wins this split.

### ✍️ Toy 7 · Ensemble choice uses risk, cost, gap, and stability

A forest is chosen by its full decision score, not just its training fit or number of trees.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)
t7_losses = np.array([0.24, 0.18, 0.30, 0.20, 0.16, 0.22])

print("validation losses:", t7_losses.tolist())  # -> [0.24, 0.18, 0.3, 0.2, 0.16, 0.22]

t7_risk = np.mean(t7_losses)

print("empirical risk:", round(float(t7_risk), 3))  # -> 0.217

t7_cost = 0.09

print("ensemble cost:", t7_cost)  # -> 0.09

t7_score = t7_risk + t7_cost

print("risk plus cost:", round(float(t7_score), 3))  # -> 0.307

t7_flexible = 0.36

print("flexible alternative:", t7_flexible)  # -> 0.36

t7_gap = t7_flexible - t7_score

print("gap:", round(float(t7_gap), 3))  # -> 0.053

t7_stable = 0.85 * t7_score

print("stabilized score:", round(float(t7_stable), 3))  # -> 0.261

t7_scores = np.array([t7_score, t7_flexible, t7_stable])

print("scores:", np.round(t7_scores, 3).tolist())  # -> [0.307, 0.36, 0.261]

assert int(np.argmin(t7_scores)) == 2

plt.figure(figsize=(4.8, 2.8))
plt.bar(["forest", "flexible", "stable"], t7_scores, color=["steelblue", "gray", "seagreen"])
plt.ylabel("lower is better")
plt.title("Toy 7 · full ensemble score")
plt.show()

▶ What you'll see: the stabilized ensemble has the lowest full score.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, bootstrap sampling, split scoring, and vectorized averaging.
import matplotlib.pyplot as plt # load Matplotlib for the small diagnostic plots in every example.
np.random.seed(0) # make all examples reproducible.

def mse(y, pred): # compute mean squared error for targets and predictions.
    y = np.asarray(y, dtype=float) # ensure numeric target array.
    pred = np.asarray(pred, dtype=float) # ensure numeric prediction array.
    return float(np.mean((y - pred) ** 2)) # average squared residuals.

def regression_stump_fit(X, y, rows=None, feature_ids=None): # fit a tiny regression stump from scratch.
    X = np.asarray(X, dtype=float) # convert features to float array.
    y = np.asarray(y, dtype=float) # convert targets to float array.
    rows = np.arange(len(y)) if rows is None else np.asarray(rows, dtype=int) # choose training rows.
    feature_ids = np.arange(X.shape[1]) if feature_ids is None else np.asarray(feature_ids, dtype=int) # choose candidate features.
    Xr, yr = X[rows], y[rows] # slice the training rows.
    best = (np.inf, int(feature_ids[0]), float(Xr[0, feature_ids[0]]), float(np.mean(yr)), float(np.mean(yr))) # initialize fallback stump.
    for j in feature_ids: # scan allowed features.
        order = np.argsort(Xr[:, j]) # order rows by the candidate feature.
        xs, ys = Xr[order, j], yr[order] # sorted feature values and targets.
        thresholds = (xs[:-1] + xs[1:]) / 2 # split points between neighboring values.
        for t in thresholds: # test every split point.
            left = xs <= t # rows sent left.
            if np.sum(left) == 0 or np.sum(~left) == 0: # require two nonempty leaves.
                continue # skip degenerate splits.
            lmean, rmean = float(np.mean(ys[left])), float(np.mean(ys[~left])) # optimal leaf means under squared loss.
            pred = np.where(left, lmean, rmean) # training predictions for this split.
            score = mse(ys, pred) # split risk.
            if score < best[0]: # keep lower-risk split.
                best = (score, int(j), float(t), lmean, rmean) # store stump parameters.
    return {"feature": best[1], "threshold": best[2], "left": best[3], "right": best[4], "train_mse": best[0]} # return model.

def regression_stump_predict(model, X): # predict from a regression stump.
    X = np.asarray(X, dtype=float) # ensure feature array.
    j = model["feature"] # selected feature.
    return np.where(X[:, j] <= model["threshold"], model["left"], model["right"]) # choose leaf mean.

def bootstrap_indices(n, seed): # draw n rows with replacement from n examples.
    rng = np.random.default_rng(seed) # local deterministic generator.
    return rng.integers(0, n, size=n) # bootstrap row ids.

def show_bar(labels, values, title, ylabel="value"): # compact bar helper.
    plt.figure(figsize=(5, 3)) # make a compact figure.
    plt.bar(labels, values, color="steelblue") # draw bars.
    plt.title(title) # title the plot.
    plt.ylabel(ylabel) # label the numeric axis.
    plt.show() # display the figure.

## 🟢 Basics (warm-up)

### Basic 1 — Average verified losses

**Goal.** Compute the empirical risk from three per-example losses, because the lesson's decision arithmetic starts with an average. We build it in 2 steps.

In [ ]:
losses_b1 = np.array([0.235, 0.135, 0.488]) # store the three verified losses from the lesson block.

print("losses:", losses_b1) # inspect the raw per-example losses.
print("sum:", round(float(np.sum(losses_b1)), 3)) # inspect the numerator before averaging.

assert round(float(np.sum(losses_b1)), 3) == 0.858 # verify the hand-computed sum.

▶ What you'll see: three small losses summing to 0.858.

In [ ]:
risk_b1 = float(np.mean(losses_b1)) # average loss over examples.

print("empirical risk R_S:", round(risk_b1, 3)) # inspect the empirical risk.

show_bar(["ex1", "ex2", "ex3", "mean"], list(losses_b1) + [risk_b1], "Basic 1: losses and their mean", "loss") # visualize mean vs examples.
assert round(risk_b1, 3) == 0.286 # verify the lesson number.

▶ What you'll see: the mean bar is the empirical quantity optimized by the model.

👀 Takeaway: bagging still begins from the same ERM habit — average losses over examples.

### Basic 2 — Add the method cost

**Goal.** Add the complexity or operational cost to the raw risk, because model selection should not rank raw fit alone. We build it in 2 steps.

In [ ]:
risk_b2 = 0.286 # use the verified empirical risk.
cost_b2 = 0.100 # use the lesson's method cost.

print("raw risk:", risk_b2, "cost:", cost_b2) # inspect the two score components.

▶ What you'll see: the cost is separate from the empirical fit.

In [ ]:
score_b2 = risk_b2 + cost_b2 # compute the full decision score.

print("decision score:", round(score_b2, 3)) # inspect risk plus cost.

show_bar(["risk", "cost", "score"], [risk_b2, cost_b2, score_b2], "Basic 2: cost changes the score", "score component") # visualize full score.
assert round(score_b2, 3) == 0.386 # verify lesson arithmetic.

▶ What you'll see: the final score is larger than the raw empirical risk because cost is included.

👀 Takeaway: the score that drives selection is raw fit plus the relevant guardrail.

### Basic 3 — Compare against a flexible alternative

**Goal.** Compute absolute and relative gaps, because a tiny score win may be too small to trust. We build it in 2 steps.

In [ ]:
score_b3 = 0.386 # stabilized baseline before extra knob.
flex_b3 = 0.434 # tempting flexible alternative.
gap_b3 = flex_b3 - score_b3 # absolute improvement of the lower score.

print("gap:", round(gap_b3, 3)) # inspect absolute difference.

assert round(gap_b3, 3) == 0.048 # verify lesson gap.

▶ What you'll see: the lower-score method wins by 0.048.

In [ ]:
relative_b3 = gap_b3 / flex_b3 # scale the gap by the alternative's score.

print("relative gap:", round(relative_b3, 3)) # inspect scale-aware improvement.

show_bar(["baseline", "flexible"], [score_b3, flex_b3], "Basic 3: lower score wins", "decision score") # compare scores.
assert round(relative_b3, 3) == 0.111 # verify relative gap.

▶ What you'll see: the absolute gap is about 11.1% of the flexible alternative.

👀 Takeaway: compare full scores and read the gap on a meaningful scale.

### Basic 4 — Stabilize a score by 20%

**Goal.** Apply the lesson's stability knob, because constrained ensembles can trade brittle variation for future reliability. We build it in 2 steps.

In [ ]:
score_b4 = 0.386 # baseline full score.
factor_b4 = 0.80 # a 20% reduction leaves 80% of the original score.

print("baseline score:", score_b4, "multiplier:", factor_b4) # inspect the ingredients.

▶ What you'll see: the stability knob is represented as a simple multiplier.

In [ ]:
stable_b4 = factor_b4 * score_b4 # compute the stabilized score.

print("stabilized score:", round(stable_b4, 3)) # inspect the reduced decision score.

show_bar(["before", "after"], [score_b4, stable_b4], "Basic 4: stabilization lowers score", "score") # visualize reduction.
assert round(stable_b4, 3) == 0.309 # verify lesson arithmetic.

▶ What you'll see: the stabilized bar is 20% lower than the baseline bar.

👀 Takeaway: a stability knob is useful only when it improves the final decision score, not just the story.

### Basic 5 — Draw one bootstrap sample

**Goal.** Sample rows with replacement, because every bagged tree is trained on a perturbed version of the training set. We build it in 2 steps.

In [ ]:
n_b5 = 10 # number of training examples.
rows_b5 = bootstrap_indices(n_b5, seed=5) # draw a bootstrap sample of row ids.
counts_b5 = np.bincount(rows_b5, minlength=n_b5) # count row multiplicities.

print("rows:", rows_b5) # inspect sampled row ids.
print("counts:", counts_b5) # inspect duplicates and omissions.

assert len(rows_b5) == n_b5 # bootstrap sample size matches the original size.

▶ What you'll see: some row ids repeat and some row ids do not appear.

In [ ]:
oob_b5 = np.where(counts_b5 == 0)[0] # identify out-of-bag rows.

print("OOB rows:", oob_b5) # inspect omitted examples.

plt.figure(figsize=(5, 3)) # create multiplicity plot.
plt.bar(np.arange(n_b5), counts_b5, color="teal") # show how many copies each row has.
plt.title("Basic 5: bootstrap row counts") # title the plot.
plt.xlabel("row id") # label rows.
plt.ylabel("copies") # label counts.
plt.show() # display the plot.
assert np.sum(counts_b5) == n_b5 # all sampled copies are accounted for.

▶ What you'll see: zero-height bars are the validation candidates for this bootstrap tree.

👀 Takeaway: bootstrap sampling creates both tree diversity and out-of-bag validation rows.

### Basic 6 — Average tree predictions

**Goal.** Compute the bagging prediction from several tree outputs, because the ensemble prediction is a mean. We build it in 2 steps.

In [ ]:
preds_b6 = np.array([1.8, 2.4, 2.1, 2.7]) # four noisy tree predictions for one example.

print("tree predictions:", preds_b6) # inspect individual base learners.
print("number of trees:", len(preds_b6)) # inspect B.

▶ What you'll see: each tree gives a different number for the same input.

In [ ]:
bag_b6 = float(np.mean(preds_b6)) # average predictions.

print("bagged prediction:", round(bag_b6, 3)) # inspect ensemble output.

plt.figure(figsize=(5, 3)) # create plot.
plt.scatter(np.arange(len(preds_b6)), preds_b6, color="gray") # individual tree predictions.
plt.axhline(bag_b6, color="crimson", label="mean") # ensemble mean.
plt.title("Basic 6: ensemble mean") # title plot.
plt.ylabel("prediction") # label predictions.
plt.legend() # show mean label.
plt.show() # display.
assert round(bag_b6, 3) == 2.25 # verify average.

▶ What you'll see: the ensemble line sits in the middle of the noisy tree predictions.

👀 Takeaway: bagging keeps the prediction scale by dividing the sum of tree outputs by B.

### Basic 7 — Measure prediction variance before averaging

**Goal.** Quantify how noisy individual tree predictions are, because bagging is mainly a variance-reduction method. We build it in 2 steps.

In [ ]:
preds_b7 = np.array([1.8, 2.4, 2.1, 2.7]) # reuse a small set of tree predictions.
mean_b7 = float(np.mean(preds_b7)) # compute center.
deviations_b7 = preds_b7 - mean_b7 # compute deviations from center.

print("deviations:", np.round(deviations_b7, 3)) # inspect noise around mean.

▶ What you'll see: some trees are above the ensemble mean and some are below it.

In [ ]:
var_b7 = float(np.var(preds_b7)) # variance of individual tree predictions.

print("prediction variance:", round(var_b7, 3)) # inspect spread.

show_bar(["T1", "T2", "T3", "T4"], np.abs(deviations_b7), "Basic 7: absolute deviations", "|tree - mean|") # visualize spread.
assert round(var_b7, 3) == 0.113 # verify variance.

▶ What you'll see: larger deviation bars are the unstable tree outputs the average smooths.

👀 Takeaway: high-variance base learners are exactly where averaging can help.

### Basic 8 — Fit one regression stump

**Goal.** Train a one-split tree by minimizing squared error, because trees are the base learners bagging averages. We build it in 2 steps.

In [ ]:
X_b8 = np.array([[0.05], [0.12], [0.20], [0.55], [0.70], [0.90]]) # one-dimensional training features.
y_b8 = np.array([0.2, 0.1, 0.3, 1.2, 1.1, 1.4]) # targets with a visible jump.
stump_b8 = regression_stump_fit(X_b8, y_b8) # fit a regression stump.

print("stump:", stump_b8) # inspect feature, threshold, leaf means, and train MSE.

assert round(stump_b8["threshold"], 3) == 0.375 # verify split location.

▶ What you'll see: the stump chooses the threshold between low and high targets.

In [ ]:
pred_b8 = regression_stump_predict(stump_b8, X_b8) # predict training rows.

print("predictions:", np.round(pred_b8, 3)) # inspect leaf-mean predictions.

plt.figure(figsize=(5, 3)) # create stump plot.
plt.scatter(X_b8[:, 0], y_b8, color="black") # plot data.
plt.step(X_b8[:, 0], pred_b8, where="mid", color="crimson") # plot piecewise-constant stump.
plt.title("Basic 8: one fitted stump") # title plot.
plt.xlabel("x") # label feature.
plt.ylabel("y") # label target.
plt.show() # display.
assert round(mse(y_b8, pred_b8), 3) == round(stump_b8["train_mse"], 3) # verify stored MSE.

▶ What you'll see: a two-level prediction rule that averages targets inside each leaf.

👀 Takeaway: a regression stump is the smallest inspectable tree used by a bagging ensemble.

### Basic 9 — Identify out-of-bag rows

**Goal.** Find rows omitted by a bootstrap sample, because those rows can validate that tree. We build it in 2 steps.

In [ ]:
n_b9 = 8 # original training size.
rows_b9 = bootstrap_indices(n_b9, seed=9) # draw bootstrap rows.
used_b9 = np.unique(rows_b9) # rows seen at least once.

print("used rows:", used_b9) # inspect in-bag row ids.

▶ What you'll see: fewer than eight unique rows may be used because sampling is with replacement.

In [ ]:
oob_b9 = np.setdiff1d(np.arange(n_b9), used_b9) # rows absent from the bootstrap.

print("OOB rows:", oob_b9) # inspect validation candidates.

show_bar(["in-bag unique", "OOB"], [len(used_b9), len(oob_b9)], "Basic 9: in-bag vs OOB counts", "rows") # visualize counts.
assert len(used_b9) + len(oob_b9) == n_b9 # every row is either used or OOB.

▶ What you'll see: out-of-bag rows are a natural holdout for this particular tree.

👀 Takeaway: OOB validation reuses training data efficiently without scoring a row on a tree that trained on it.

### Basic 10 — Choose a random feature subset

**Goal.** Restrict a split to a random subset of features, because random forests lower correlation by diversifying split choices. We build it in 2 steps.

In [ ]:
features_b10 = np.arange(5) # five available feature ids.
rng_b10 = np.random.default_rng(10) # deterministic local generator.
offered_b10 = rng_b10.choice(features_b10, size=2, replace=False) # choose mtry=2 candidate features.

print("all features:", features_b10) # inspect full set.
print("offered features:", offered_b10) # inspect random subset.

assert len(np.unique(offered_b10)) == 2 # no duplicate features are offered.

▶ What you'll see: the split sees only two of the five possible features.

In [ ]:
mask_b10 = np.isin(features_b10, offered_b10) # mark offered features.
plt.figure(figsize=(5, 3)) # create feature-subset plot.
plt.bar(features_b10, mask_b10.astype(int), color="purple") # offered = 1, hidden = 0.
plt.title("Basic 10: random feature subset") # title plot.
plt.xlabel("feature id") # label features.
plt.ylabel("offered to split") # label indicator.
plt.show() # display.
assert np.sum(mask_b10) == 2 # exactly two features were offered.

▶ What you'll see: some features are deliberately unavailable to this split.

👀 Takeaway: random forests add feature randomness so trees are less correlated than ordinary bagged trees.

## 🟡 Easy

### Easy 1 — Train a small bagged-stump regressor

**Goal.** Fit many bootstrap stumps and average them, because this is bagging in its most inspectable regression form. We build it in 3 steps.

In [ ]:
X_e1 = np.linspace(0, 1, 12)[:, None] # one feature on a regular grid.
y_e1 = np.sin(2 * np.pi * X_e1[:, 0]) + 0.15 * np.array([0, 1, -1, 1, 0, -1, 1, 0, -1, 1, -1, 0]) # wavy target with fixed noise.
B_e1 = 30 # number of bootstrap stumps.

print("data shape:", X_e1.shape, "trees:", B_e1) # inspect training setup.

▶ What you'll see: a tiny one-feature regression problem with 30 bootstrap trees.

In [ ]:
models_e1 = [] # store fitted stumps.
for b_e1 in range(B_e1): # train each tree.
    rows_e1 = bootstrap_indices(len(y_e1), seed=100 + b_e1) # draw bootstrap rows.
    models_e1.append(regression_stump_fit(X_e1, y_e1, rows=rows_e1)) # fit and save stump.
preds_e1 = np.array([regression_stump_predict(m_e1, X_e1) for m_e1 in models_e1]) # collect tree predictions.
bag_pred_e1 = np.mean(preds_e1, axis=0) # average over trees.

print("single tree MSE:", round(mse(y_e1, preds_e1[0]), 3), "bagged MSE:", round(mse(y_e1, bag_pred_e1), 3)) # compare.

▶ What you'll see: the bagged fit is usually smoother and lower-error than one random stump.

In [ ]:
plt.figure(figsize=(5, 3)) # create ensemble plot.
plt.scatter(X_e1[:, 0], y_e1, color="black", label="data") # plot observations.
plt.plot(X_e1[:, 0], preds_e1[0], color="gray", alpha=0.7, label="one stump") # plot one base learner.
plt.plot(X_e1[:, 0], bag_pred_e1, color="crimson", linewidth=2, label="bagged mean") # plot ensemble.
plt.title("Easy 1: bagging stumps") # title plot.
plt.legend() # show labels.
plt.show() # display.
assert bag_pred_e1.shape == y_e1.shape # prediction has one value per row.

▶ What you'll see: the red averaged curve is less jumpy than one gray bootstrap stump.

👀 Takeaway: bagging turns many unstable tree rules into one steadier regression rule.

### Easy 2 — Compute out-of-bag MSE

**Goal.** Estimate validation error from OOB predictions, because each bootstrap tree leaves some rows untouched. We build it in 3 steps.

In [ ]:
X_e2 = np.linspace(0, 1, 14)[:, None] # one-feature regression grid.
y_e2 = np.cos(2 * np.pi * X_e2[:, 0]) + 0.1 * np.array([1, -1, 0, 1, 0, -1, 1, 0, -1, 0, 1, -1, 0, 1]) # deterministic target.
B_e2 = 40 # number of bootstrap stumps.
oob_sum_e2 = np.zeros(len(y_e2)) # accumulated OOB predictions.
oob_count_e2 = np.zeros(len(y_e2)) # OOB prediction counts.

print("rows:", len(y_e2), "trees:", B_e2) # inspect setup.

▶ What you'll see: every row can receive predictions from trees that skipped it.

In [ ]:
for b_e2 in range(B_e2): # train and OOB-score each bootstrap stump.
    rows_e2 = bootstrap_indices(len(y_e2), seed=200 + b_e2) # bootstrap sample.
    model_e2 = regression_stump_fit(X_e2, y_e2, rows=rows_e2) # fit stump.
    oob_e2 = np.setdiff1d(np.arange(len(y_e2)), np.unique(rows_e2)) # find rows not used.
    if len(oob_e2) > 0: # score only omitted rows.
        oob_sum_e2[oob_e2] += regression_stump_predict(model_e2, X_e2[oob_e2]) # add OOB preds.
        oob_count_e2[oob_e2] += 1 # count OOB predictions.

print("min OOB count:", int(np.min(oob_count_e2)), "max OOB count:", int(np.max(oob_count_e2))) # inspect coverage.

assert np.min(oob_count_e2) > 0 # all rows have OOB predictions for this setup.

▶ What you'll see: different rows have different numbers of OOB predictions.

In [ ]:
oob_pred_e2 = oob_sum_e2 / oob_count_e2 # average OOB predictions.
oob_mse_e2 = mse(y_e2, oob_pred_e2) # compute validation-like MSE.

print("OOB MSE:", round(oob_mse_e2, 3)) # inspect OOB error.

plt.figure(figsize=(5, 3)) # create OOB plot.
plt.scatter(X_e2[:, 0], y_e2, color="black", label="truth") # plot true targets.
plt.scatter(X_e2[:, 0], oob_pred_e2, color="orange", marker="x", label="OOB pred") # plot OOB predictions.
plt.title("Easy 2: out-of-bag predictions") # title plot.
plt.legend() # show labels.
plt.show() # display.
assert oob_mse_e2 >= 0 # MSE is nonnegative.

▶ What you'll see: OOB predictions are a validation estimate built from the bootstrap leftovers.

👀 Takeaway: OOB scoring keeps the generalization warning visible while using the training set efficiently.

### Easy 3 — Compare bagging variance as B grows

**Goal.** Track how prediction variability changes with the number of trees, because averaging more noisy learners should stabilize the ensemble. We build it in 3 steps.

In [ ]:
rng_e3 = np.random.default_rng(3) # deterministic noise generator.
base_signal_e3 = 2.0 # shared signal all trees try to estimate.
noise_e3 = rng_e3.normal(0, 1.0, size=200) # many simulated tree errors.
tree_preds_e3 = base_signal_e3 + noise_e3 # simulated tree predictions.

print("first five tree predictions:", np.round(tree_preds_e3[:5], 3)) # inspect raw base learners.

▶ What you'll see: each tree prediction is the same signal plus different noise.

In [ ]:
B_grid_e3 = np.array([1, 2, 5, 10, 25, 50, 100]) # ensemble sizes to test.
ensemble_preds_e3 = np.array([np.mean(tree_preds_e3[:B_e3]) for B_e3 in B_grid_e3]) # running means.
errors_e3 = np.abs(ensemble_preds_e3 - base_signal_e3) # absolute error from true signal.

print("running means:", np.round(ensemble_preds_e3, 3)) # inspect stabilization.
print("absolute errors:", np.round(errors_e3, 3)) # inspect accuracy.

▶ What you'll see: the running mean usually moves closer to the signal as B grows.

In [ ]:
plt.figure(figsize=(5, 3)) # create B sweep plot.
plt.plot(B_grid_e3, errors_e3, marker="o", color="seagreen") # plot error vs ensemble size.
plt.title("Easy 3: averaging more trees stabilizes") # title plot.
plt.xlabel("B trees") # label ensemble size.
plt.ylabel("|bagged mean - signal|") # label error.
plt.show() # display.
assert len(errors_e3) == len(B_grid_e3) # one error per B.

▶ What you'll see: the curve is not perfectly monotone for one random draw, but larger B reduces volatility overall.

👀 Takeaway: bagging is a variance play, so its benefit appears as the average becomes less sensitive to one tree.

### Easy 4 — Random-forest split with mtry

**Goal.** Fit stumps while exposing only one random feature per tree, because random forests reduce tree correlation through feature subsampling. We build it in 3 steps.

In [ ]:
X_e4 = np.array([[0.0, 0.1], [0.1, 0.2], [0.2, 0.0], [0.8, 0.9], [0.9, 0.7], [1.0, 0.8]]) # two-feature data.
y_e4 = np.array([0., 0., 0., 1., 1., 1.]) # binary target treated as numeric probability.
B_e4 = 12 # number of random-feature stumps.

print("features:", X_e4.shape[1], "trees:", B_e4) # inspect setup.

▶ What you'll see: each tree will choose from a two-feature table.

In [ ]:
features_used_e4 = [] # record chosen split features.
models_e4 = [] # store stumps.
for b_e4 in range(B_e4): # train random-feature stumps.
    rows_e4 = bootstrap_indices(len(y_e4), seed=400 + b_e4) # bootstrap rows.
    rng_e4 = np.random.default_rng(500 + b_e4) # deterministic feature sampler.
    feat_e4 = rng_e4.choice(np.arange(X_e4.shape[1]), size=1, replace=False) # mtry=1 feature subset.
    model_e4 = regression_stump_fit(X_e4, y_e4, rows=rows_e4, feature_ids=feat_e4) # fit constrained stump.
    models_e4.append(model_e4) # save model.
    features_used_e4.append(model_e4["feature"]) # record actual split feature.

print("features used:", features_used_e4) # inspect split diversity.

assert set(features_used_e4).issubset({0, 1}) # only valid features appear.

▶ What you'll see: different trees use different feature ids.

In [ ]:
votes_e4 = np.mean([regression_stump_predict(m_e4, X_e4) for m_e4 in models_e4], axis=0) # average probability-like outputs.

print("forest predictions:", np.round(votes_e4, 3)) # inspect ensemble probabilities.

plt.figure(figsize=(5, 3)) # create feature-use plot.
plt.bar(["feature 0", "feature 1"], np.bincount(features_used_e4, minlength=2), color="purple") # count feature usage.
plt.title("Easy 4: random feature usage") # title plot.
plt.ylabel("trees") # label count.
plt.show() # display.
assert votes_e4.shape == y_e4.shape # one prediction per row.

▶ What you'll see: feature usage is spread across the forest instead of locked to one greedy feature.

👀 Takeaway: mtry intentionally weakens individual split greediness to make the averaged forest stronger.

### Easy 5 — Pick the lowest full score

**Goal.** Choose among baseline, flexible, and stabilized settings using full decision scores, because the lesson's final decision is a minimum over comparable quantities. We build it in 3 steps.

In [ ]:
names_e5 = np.array(["baseline", "flexible", "stabilized"]) # candidate methods.
scores_e5 = np.array([0.386, 0.434, 0.309]) # verified full decision scores.

print("scores:", dict(zip(names_e5, scores_e5))) # inspect candidates.

▶ What you'll see: all candidates are on the same lower-is-better scale.

In [ ]:
best_idx_e5 = int(np.argmin(scores_e5)) # choose the lowest score.
best_name_e5 = names_e5[best_idx_e5] # read candidate name.
best_score_e5 = float(scores_e5[best_idx_e5]) # read candidate score.

print("best:", best_name_e5, "score:", best_score_e5) # inspect selection.

assert best_name_e5 == "stabilized" and round(best_score_e5, 3) == 0.309 # verify lesson decision.

▶ What you'll see: the stabilized candidate wins the toy selection.

In [ ]:
plt.figure(figsize=(5, 3)) # create comparison plot.
colors_e5 = ["gray", "orange", "seagreen"] # color candidates.
plt.bar(names_e5, scores_e5, color=colors_e5) # plot full scores.
plt.title("Easy 5: final comparable scores") # title plot.
plt.ylabel("lower is better") # label decision score.
plt.show() # display.
assert np.min(scores_e5) == best_score_e5 # selection matches numeric minimum.

▶ What you'll see: the lowest bar is the one selected.

👀 Takeaway: the correct unit of judgment is the full score, not the prettiest training fragment.

## 🔴 Advanced

### Advanced 1 — Estimate variance reduction empirically

**Goal.** Simulate many ensembles and compare their variance to single trees, because bagging's central claim is reduced prediction variance. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(11) # deterministic simulator.
trials_a1 = 500 # number of repeated experiments.
B_a1 = 20 # trees per ensemble.
single_a1 = rng_a1.normal(0, 1, size=trials_a1) # one noisy tree per trial.

print("trials:", trials_a1, "B:", B_a1) # inspect simulation size.

▶ What you'll see: the experiment compares one tree to an average of 20 noisy trees.

In [ ]:
noise_a1 = rng_a1.normal(0, 1, size=(trials_a1, B_a1)) # independent tree errors.
bagged_a1 = np.mean(noise_a1, axis=1) # one ensemble average per trial.
var_single_a1 = float(np.var(single_a1)) # empirical variance of one tree.
var_bag_a1 = float(np.var(bagged_a1)) # empirical variance of B-tree average.

print("single variance:", round(var_single_a1, 3), "bagged variance:", round(var_bag_a1, 3)) # inspect variance reduction.

▶ What you'll see: the bagged variance is much smaller than the single-tree variance.

In [ ]:
ratio_a1 = var_bag_a1 / var_single_a1 # empirical variance factor.
theory_a1 = 1 / B_a1 # independent-tree variance factor.

print("empirical ratio:", round(ratio_a1, 3), "theory 1/B:", round(theory_a1, 3)) # compare simulation and theory.

assert ratio_a1 < 0.1 # with B=20, the variance factor should be near .05.

▶ What you'll see: the empirical ratio is close to the independent-noise prediction.

In [ ]:
plt.figure(figsize=(5, 3)) # create histogram comparison.
plt.hist(single_a1, bins=30, alpha=0.5, label="single tree") # distribution of one tree.
plt.hist(bagged_a1, bins=30, alpha=0.7, label="bagged mean") # distribution of ensemble mean.
plt.title("Advanced 1: bagging narrows prediction spread") # title plot.
plt.legend() # show labels.
plt.show() # display.

▶ What you'll see: the bagged distribution is visibly narrower around zero.

👀 Takeaway: when tree errors are weakly correlated, averaging sharply reduces prediction variance.

### Advanced 2 — Show correlation creates a variance floor

**Goal.** Simulate correlated tree errors, because random forests exist to lower correlation as well as average many trees. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(12) # deterministic simulator.
trials_a2 = 600 # repeated ensembles.
B_a2 = 40 # trees per ensemble.
rhos_a2 = np.array([0.0, 0.2, 0.6]) # correlation levels.

print("rho grid:", rhos_a2) # inspect correlation settings.

▶ What you'll see: the sweep compares independent through highly correlated trees.

In [ ]:
ratios_a2 = [] # store variance factors.
for rho_a2 in rhos_a2: # simulate one correlation level.
    shared_a2 = rng_a2.normal(0, 1, size=(trials_a2, 1)) # shared ensemble-level noise.
    private_a2 = rng_a2.normal(0, 1, size=(trials_a2, B_a2)) # tree-specific noise.
    errors_a2 = np.sqrt(rho_a2) * shared_a2 + np.sqrt(1 - rho_a2) * private_a2 # correlated tree errors.
    means_a2 = np.mean(errors_a2, axis=1) # bagged prediction error.
    ratios_a2.append(float(np.var(means_a2))) # single-tree variance is about 1.

print("variance factors:", np.round(ratios_a2, 3)) # inspect remaining variance.

▶ What you'll see: higher correlation leaves much more variance after averaging.

In [ ]:
theory_a2 = rhos_a2 + (1 - rhos_a2) / B_a2 # correlated-average formula.

print("theory:", np.round(theory_a2, 3)) # inspect predicted variance factors.

assert np.all(np.abs(np.array(ratios_a2) - theory_a2) < 0.08) # simulation should match formula roughly.

▶ What you'll see: simulated variance follows the correlation-floor formula.

In [ ]:
plt.figure(figsize=(5, 3)) # create comparison plot.
plt.plot(rhos_a2, ratios_a2, marker="o", label="simulated") # empirical factors.
plt.plot(rhos_a2, theory_a2, marker="s", label="formula") # theoretical factors.
plt.title("Advanced 2: correlation limits averaging") # title plot.
plt.xlabel("average tree-error correlation") # label rho.
plt.ylabel("variance factor") # label variance.
plt.legend() # show labels.
plt.show() # display.

▶ What you'll see: the curve rises with correlation even though B is fixed at 40.

👀 Takeaway: random forests help because feature randomness can reduce tree correlation before averaging.

### Advanced 3 — Compare bagging and random-feature forests

**Goal.** Train two stump ensembles with and without feature subsampling, because the forest version should use more diverse split features. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(13) # deterministic data generator.
X_a3 = rng_a3.normal(size=(80, 4)) # four input features.
y_a3 = 2 * X_a3[:, 0] - 1.5 * X_a3[:, 1] + 0.3 * rng_a3.normal(size=80) # target depends mainly on features 0 and 1.
B_a3 = 60 # ensemble size.

print("X shape:", X_a3.shape, "trees:", B_a3) # inspect problem size.

▶ What you'll see: a small regression problem with four possible split features.

In [ ]:
bag_features_a3 = [] # split features for ordinary bagging.
rf_features_a3 = [] # split features for random feature stumps.
bag_models_a3 = [] # ordinary bagged stumps.
rf_models_a3 = [] # random-feature stumps.
for b_a3 in range(B_a3): # train both ensembles with matching bootstrap rows.
    rows_a3 = bootstrap_indices(len(y_a3), seed=600 + b_a3) # bootstrap sample.
    bag_m_a3 = regression_stump_fit(X_a3, y_a3, rows=rows_a3) # all features available.
    rng_feat_a3 = np.random.default_rng(700 + b_a3) # feature sampler.
    offered_a3 = rng_feat_a3.choice(np.arange(X_a3.shape[1]), size=2, replace=False) # mtry=2.
    rf_m_a3 = regression_stump_fit(X_a3, y_a3, rows=rows_a3, feature_ids=offered_a3) # constrained split.
    bag_models_a3.append(bag_m_a3); rf_models_a3.append(rf_m_a3) # save models.
    bag_features_a3.append(bag_m_a3["feature"]); rf_features_a3.append(rf_m_a3["feature"]) # save features.

print("bag feature counts:", np.bincount(bag_features_a3, minlength=4)) # inspect greedy feature concentration.
print("RF feature counts:", np.bincount(rf_features_a3, minlength=4)) # inspect random-feature diversity.

▶ What you'll see: ordinary bagging concentrates more heavily on the strongest features.

In [ ]:
bag_pred_a3 = np.mean([regression_stump_predict(m_a3, X_a3) for m_a3 in bag_models_a3], axis=0) # ordinary ensemble predictions.
rf_pred_a3 = np.mean([regression_stump_predict(m_a3, X_a3) for m_a3 in rf_models_a3], axis=0) # random-feature ensemble predictions.

print("bag MSE:", round(mse(y_a3, bag_pred_a3), 3), "RF MSE:", round(mse(y_a3, rf_pred_a3), 3)) # compare fit.

assert bag_pred_a3.shape == rf_pred_a3.shape == y_a3.shape # both predict all rows.

▶ What you'll see: both ensembles are valid; the feature-random one trades split optimality for diversity.

In [ ]:
x_a3 = np.arange(4) # feature ids.
plt.figure(figsize=(5, 3)) # create grouped feature-count plot.
plt.bar(x_a3 - 0.18, np.bincount(bag_features_a3, minlength=4), width=0.36, label="bagging") # ordinary counts.
plt.bar(x_a3 + 0.18, np.bincount(rf_features_a3, minlength=4), width=0.36, label="random features") # RF counts.
plt.title("Advanced 3: split-feature diversity") # title plot.
plt.xlabel("feature id") # label feature.
plt.ylabel("trees using feature") # label count.
plt.legend() # show labels.
plt.show() # display.

▶ What you'll see: the random-feature ensemble spreads split choices across more features.

👀 Takeaway: random forests deliberately decorrelate trees by preventing every split from making the same greedy feature choice.

### Advanced 4 — Build a full OOB learning curve

**Goal.** Track OOB error as more trees are added, because forest size is chosen by watching validation-like performance flatten. We build it in 4 steps.

In [ ]:
X_a4 = np.linspace(-1, 1, 30)[:, None] # one-dimensional regression inputs.
y_a4 = X_a4[:, 0] ** 2 + 0.1 * np.sin(8 * X_a4[:, 0]) # nonlinear target.
B_a4 = 80 # maximum forest size.
checkpoints_a4 = np.array([5, 10, 20, 40, 80]) # ensemble sizes to report.

print("checkpoints:", checkpoints_a4) # inspect learning-curve points.

▶ What you'll see: OOB error will be measured at several forest sizes.

In [ ]:
oob_sum_a4 = np.zeros((len(checkpoints_a4), len(y_a4))) # OOB sums per checkpoint.
oob_count_a4 = np.zeros((len(checkpoints_a4), len(y_a4))) # OOB counts per checkpoint.
models_a4 = [] # store models as trees arrive.
for b_a4 in range(B_a4): # train trees one by one.
    rows_a4 = bootstrap_indices(len(y_a4), seed=800 + b_a4) # bootstrap rows.
    model_a4 = regression_stump_fit(X_a4, y_a4, rows=rows_a4) # fit stump.
    models_a4.append(model_a4) # save model.
    oob_a4 = np.setdiff1d(np.arange(len(y_a4)), np.unique(rows_a4)) # rows omitted by this tree.
    for c_idx_a4, c_a4 in enumerate(checkpoints_a4): # update checkpoints that include this tree.
        if b_a4 < c_a4 and len(oob_a4) > 0: # tree belongs to this checkpoint.
            oob_sum_a4[c_idx_a4, oob_a4] += regression_stump_predict(model_a4, X_a4[oob_a4]) # add OOB prediction.
            oob_count_a4[c_idx_a4, oob_a4] += 1 # count OOB prediction.

print("min OOB counts by checkpoint:", np.min(oob_count_a4, axis=1).astype(int)) # inspect coverage.

▶ What you'll see: larger checkpoints have better OOB coverage for every row.

In [ ]:
oob_mse_a4 = [] # store OOB MSE per checkpoint.
for c_idx_a4 in range(len(checkpoints_a4)): # compute each checkpoint's error.
    valid_a4 = oob_count_a4[c_idx_a4] > 0 # rows with at least one OOB prediction.
    pred_a4 = np.zeros_like(y_a4) # initialize predictions.
    pred_a4[valid_a4] = oob_sum_a4[c_idx_a4, valid_a4] / oob_count_a4[c_idx_a4, valid_a4] # average OOB preds.
    oob_mse_a4.append(mse(y_a4[valid_a4], pred_a4[valid_a4])) # score covered rows.

print("OOB MSE curve:", np.round(oob_mse_a4, 4)) # inspect learning curve.

assert len(oob_mse_a4) == len(checkpoints_a4) # one score per checkpoint.

▶ What you'll see: OOB MSE usually stabilizes as the ensemble gets large.

In [ ]:
plt.figure(figsize=(5, 3)) # create learning curve plot.
plt.plot(checkpoints_a4, oob_mse_a4, marker="o", color="crimson") # plot OOB error vs B.
plt.title("Advanced 4: OOB error vs forest size") # title plot.
plt.xlabel("trees B") # label ensemble size.
plt.ylabel("OOB MSE") # label validation error.
plt.show() # display.

▶ What you'll see: the curve shows whether adding trees still buys validation improvement.

👀 Takeaway: choose a forest size where OOB performance has mostly flattened, not where training fit looks best.

### Advanced 5 — Tune stability with a full decision score

**Goal.** Combine empirical risk, cost, and stabilization into one comparable score, because the lesson warns against optimizing raw training fragments. We build it in 4 steps.

In [ ]:
risk_a5 = np.array([0.286, 0.270, 0.260, 0.255]) # raw risks for increasingly flexible settings.
cost_a5 = np.array([0.100, 0.140, 0.190, 0.250]) # complexity or operational costs.
stability_a5 = np.array([1.00, 0.92, 0.86, 0.80]) # multiplicative stability adjustments.
labels_a5 = np.array(["base", "mild", "strong", "very strong"]) # setting names.

print("raw risks:", risk_a5) # inspect raw fit.
print("costs:", cost_a5) # inspect guardrails.

▶ What you'll see: raw risk improves as flexibility rises, but cost also rises.

In [ ]:
pre_score_a5 = risk_a5 + cost_a5 # score before stability adjustment.
final_score_a5 = stability_a5 * pre_score_a5 # full comparable decision score.

print("pre-stability scores:", np.round(pre_score_a5, 3)) # inspect raw plus cost.
print("final scores:", np.round(final_score_a5, 3)) # inspect full decision score.

assert round(float(final_score_a5[0]), 3) == 0.386 # baseline reproduces the lesson score.

▶ What you'll see: the setting with best raw risk is not automatically the final winner.

In [ ]:
best_idx_a5 = int(np.argmin(final_score_a5)) # choose the lowest full score.

print("best setting:", labels_a5[best_idx_a5], "score:", round(float(final_score_a5[best_idx_a5]), 3)) # inspect selected setting.

assert final_score_a5[best_idx_a5] == np.min(final_score_a5) # verify correct selection.

▶ What you'll see: selection is based on the complete lower-is-better score.

In [ ]:
x_a5 = np.arange(len(labels_a5)) # positions for grouped bars.
plt.figure(figsize=(6, 3)) # create grouped score plot.
plt.bar(x_a5 - 0.18, risk_a5, width=0.36, label="raw risk") # plot raw risk.
plt.bar(x_a5 + 0.18, final_score_a5, width=0.36, label="full score") # plot full score.
plt.xticks(x_a5, labels_a5, rotation=15) # label settings.
plt.title("Advanced 5: raw fit vs full decision score") # title plot.
plt.ylabel("lower is better") # label score scale.
plt.legend() # show labels.
plt.show() # display.

▶ What you'll see: raw-risk bars can fall while full-score bars reveal the true selection tradeoff.

👀 Takeaway: bagging and random forests are selected by validated, cost-aware stability — not by training loss alone.